# Shared Evaluation — Next-Day Wildfire Spread 

**ALL FOUR MODELS MUST USE THIS NOTEBOOK** to compute and report metrics. This guarantees the exact same metric definitions, the same handling of uncertain (`-1`) pixels, and the same threshold are used for every model, so the comparison in Section 7 of the report is fair and apples-to-apples.

**Quick usage (from your own model notebook, after training):**
```python
%run ../shared/evaluate.ipynb

results = evaluate_model(model, test_loader, device, model_name="unet")
save_results(results, "../results/unet_metrics.json")
plot_prediction_grid(model, test_loader, device, model_name="unet", n_samples=5)
```

At the end (Week 3), one member runs `compare_models()` over all 4 saved results JSON files to build the final comparison table/plots.


## Imports

In [ ]:
import json
import os
from typing import Dict, List, Optional

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)
import matplotlib.pyplot as plt
import pandas as pd

DEFAULT_THRESHOLD = 0.5  # agree as a team before changing; must be the same for all 4 models


## 1. Core metric computation

Computes the full metric set — accuracy, precision, recall, F1, ROC-AUC, IoU, and confusion matrix — on **valid pixels only** (excluding the `-1` uncertain sentinel).

In [ ]:
def _iou_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Intersection-over-Union for the positive (fire) class."""
    intersection = np.logical_and(y_true == 1, y_pred == 1).sum()
    union = np.logical_or(y_true == 1, y_pred == 1).sum()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union


def compute_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    valid_mask: np.ndarray,
    threshold: float = DEFAULT_THRESHOLD,
) -> Dict[str, float]:
    """
    Args:
        y_true: ground-truth fire mask, any shape, values in {0, 1, -1}
        y_prob: predicted fire probability (post-sigmoid), same shape, in [0, 1]
        valid_mask: same shape, 1.0 = valid pixel, 0.0 = uncertain (-1) pixel to exclude
        threshold: probability cutoff for turning y_prob into a binary prediction

    Returns:
        dict with accuracy, precision, recall, f1, roc_auc, iou, and the
        raw confusion matrix (as nested list, for JSON serialization)
    """
    valid = valid_mask.astype(bool).flatten()
    yt = y_true.flatten()[valid]
    yp_prob = y_prob.flatten()[valid]
    yp = (yp_prob >= threshold).astype(np.int32)
    yt = yt.astype(np.int32)

    metrics = {
        "accuracy": float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, zero_division=0)),
        "recall": float(recall_score(yt, yp, zero_division=0)),
        "f1": float(f1_score(yt, yp, zero_division=0)),
        "iou": float(_iou_score(yt, yp)),
    }

    # ROC-AUC needs both classes present; guard against a degenerate batch
    if len(np.unique(yt)) > 1:
        metrics["roc_auc"] = float(roc_auc_score(yt, yp_prob))
    else:
        metrics["roc_auc"] = float("nan")

    cm = confusion_matrix(yt, yp, labels=[0, 1])
    metrics["confusion_matrix"] = cm.tolist()  # [[TN, FP], [FN, TP]]

    return metrics


## 2. Running a model over the full test set

Runs `model` over the entire `test_loader`, applies sigmoid, and aggregates metrics across all batches. Optionally records training time and parameter count for the "computational efficiency" and "model complexity" comparisons the rubric asks for.

In [ ]:
@torch.no_grad()
def evaluate_model(
    model: torch.nn.Module,
    test_loader,
    device: str,
    model_name: str,
    threshold: float = DEFAULT_THRESHOLD,
    training_time_sec: Optional[float] = None,
    num_params: Optional[int] = None,
) -> Dict:
    """
    Assumes the model outputs raw logits of shape (B, 1, 64, 64) — sigmoid is
    applied here. If your model already applies sigmoid internally, remove
    the torch.sigmoid() call below.

    Args:
        model: trained PyTorch model, already .to(device) and .eval()
        test_loader: shared test DataLoader from shared/preprocessing.ipynb
        device: "cuda" or "cpu"
        model_name: e.g. "cnn_baseline", "unet", "resnet_unet", "attention_unet"
        threshold: probability cutoff, must match what the other 3 models use
        training_time_sec: optional, e.g. time.time() delta around your training loop
        num_params: optional, e.g. sum(p.numel() for p in model.parameters())
    """
    model.eval()
    all_true, all_prob, all_mask = [], [], []

    for x, y, mask in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_true.append(y.numpy())
        all_prob.append(probs)
        all_mask.append(mask.numpy())

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)
    valid_mask = np.concatenate(all_mask, axis=0)

    metrics = compute_metrics(y_true, y_prob, valid_mask, threshold=threshold)

    result = {"model_name": model_name, "threshold": threshold, "metrics": metrics}
    if training_time_sec is not None:
        result["training_time_sec"] = training_time_sec
    if num_params is not None:
        result["num_params"] = num_params

    return result


def save_results(result: Dict, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"Saved results to {path}")


def load_results(path: str) -> Dict:
    with open(path) as f:
        return json.load(f)


## 3. Visualizations

Each member uses these on their own trained model.

In [ ]:
def plot_confusion_matrix(cm: List[List[int]], model_name: str, ax=None):
    """cm is [[TN, FP], [FN, TP]] as produced by compute_metrics()."""
    cm = np.array(cm)
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No Fire", "Fire"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No Fire", "Fire"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {model_name}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax


def plot_training_curves(history: Dict[str, List[float]], model_name: str):
    """history example: {"train_loss": [...], "val_loss": [...]}"""
    fig, ax = plt.subplots(figsize=(6, 4))
    for key, values in history.items():
        ax.plot(values, label=key)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title(f"Training Curves — {model_name}")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
@torch.no_grad()
def plot_prediction_grid(
    model: torch.nn.Module,
    test_loader,
    device: str,
    model_name: str,
    n_samples: int = 5,
    threshold: float = DEFAULT_THRESHOLD,
):
    """Shows n_samples rows of: previous-day fire mask | ground truth next-day |
    predicted next-day (binary) | predicted probability heatmap.

    This is the single most convincing visual for both the report's Results
    section and the viva demo.
    """
    model.eval()
    x, y, mask = next(iter(test_loader))
    x = x.to(device)
    probs = torch.sigmoid(model(x)).cpu().numpy()
    preds = (probs >= threshold).astype(np.float32)

    prev_fire_channel = 11  # index of PrevFireMask in INPUT_FEATURES, see shared/preprocessing.ipynb

    n_samples = min(n_samples, x.shape[0])
    fig, axes = plt.subplots(n_samples, 4, figsize=(12, 3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Previous Day (input)", "Ground Truth (next day)", "Prediction (binary)", "Prediction (probability)"]
    for i in range(n_samples):
        prev = x[i, prev_fire_channel].cpu().numpy()
        true_mask = y[i, 0].numpy()
        pred_mask = preds[i, 0]
        pred_prob = probs[i, 0]

        for j, img in enumerate([prev, true_mask, pred_mask, pred_prob]):
            axes[i, j].imshow(img, cmap="hot", vmin=0, vmax=1)
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=10)

    fig.suptitle(f"Sample Predictions — {model_name}", fontsize=13)
    plt.tight_layout()
    plt.show()


## 4. Cross-model comparison

Run this **once, after all 4 members have finished** and saved their results JSON files to `results/`. Produces the comparison table and bar chart for Section 7 of the report.

In [ ]:
def compare_models(results_paths: Dict[str, str]) -> pd.DataFrame:
    """
    Args:
        results_paths: e.g. {
            "CNN Baseline": "../results/cnn_baseline_metrics.json",
            "U-Net": "../results/unet_metrics.json",
            "ResNet-UNet": "../results/resnet_unet_metrics.json",
            "Attention U-Net": "../results/attention_unet_metrics.json",
        }

    Returns:
        a pandas DataFrame with one row per model, ready to paste into the
        report's Results and Model Comparison section (Section 7).
    """
    rows = []
    for display_name, path in results_paths.items():
        r = load_results(path)
        row = {"Model": display_name, **r["metrics"]}
        row.pop("confusion_matrix", None)  # not tabular-friendly, plot separately
        if "training_time_sec" in r:
            row["Training Time (s)"] = r["training_time_sec"]
        if "num_params" in r:
            row["Params"] = r["num_params"]
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Model")
    return df


def plot_metric_comparison(df: pd.DataFrame, metrics: Optional[List[str]] = None):
    """Bar chart comparing all 4 models across the chosen metrics."""
    if metrics is None:
        metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "iou"]
    df[metrics].plot(kind="bar", figsize=(10, 5))
    plt.ylabel("Score")
    plt.title("Model Comparison")
    plt.xticks(rotation=0)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## Example: end-to-end usage in a model notebook

Every model notebook must run both shared notebooks before training or evaluation:

```python
# --- inside models/cnn_baseline.ipynb, models/unet.ipynb, etc. ---
%run ../shared/preprocessing.ipynb
%run ../shared/evaluate.ipynb

# The preprocessing notebook provides the same loaders to every model.
train_loader, val_loader, test_loader, pos_weight = get_dataloaders(
    data_dir=DEFAULT_DATA_DIR,
    batch_size=32,
    augment_train=True,
    num_workers=0,
)

# Build and train your model. Use val_loader for model selection.
# model = YourModel(in_channels=NUM_CHANNELS, out_channels=1).to(device)
# ... train model, track history = {"train_loss": [...], "val_loss": [...]} ...

plot_training_curves(history, model_name="your_model")

# Use test_loader only once for final evaluation.
results = evaluate_model(
    model, test_loader, device, model_name="your_model",
    training_time_sec=elapsed_time,
    num_params=sum(p.numel() for p in model.parameters()),
)
save_results(results, "../results/your_model_metrics.json")
plot_confusion_matrix(results["metrics"]["confusion_matrix"], model_name="Your Model")
plot_prediction_grid(model, test_loader, device, model_name="Your Model", n_samples=5)
```

**Then, once all 4 models are done:**

```python
# --- inside a final comparison notebook ---
%run ../shared/evaluate.ipynb

df = compare_models({
    "CNN Baseline": "../results/cnn_baseline_metrics.json",
    "U-Net": "../results/unet_metrics.json",
    "ResNet-UNet": "../results/resnet_unet_metrics.json",
    "Attention U-Net": "../results/attention_unet_metrics.json",
})
display(df)
plot_metric_comparison(df)
```

Do not tune on `test_loader`. The test set is reserved for the final report comparison.